# Hysteresis test (L=4) — first vs second order

Two directed NQS sweeps per line: **forward** (warm-started up from the topological phase) and **backward** (down from the polarized phase). Read against `notes/hysteresis_schematic.png`:

* **First order** → the branches *separate*: a **loop** in the order parameter (top row) and a **crossing** in $E/N$ (bottom row) over a coexistence window.
* **Second order** → forward and backward **coincide**: no loop, no crossing.

Data: `nersc/submit_hysteresis.sh` → `check_convergence.py --dump` per branch. The dumped `mag` (=$\langle M_x\rangle$ for the hx-sweep, $\langle M_z\rangle$ for the hz-sweep) is the conjugate magnetization — its jump *is* the order-parameter loop.

In [ ]:
# ====================== 1 · CONFIG ======================
import json, os
import numpy as np
import matplotlib.pyplot as plt

ROOT = "/Users/sanzhar123/Desktop/Approximate-Symmetries-TC-main/results"
L    = 4

# each experiment: the two branch dumps + the expected order (for labelling only)
EXP = {
    "hx-sweep  (hz=0)":   dict(field="hx", order="first?",
        fwd=f"{ROOT}/hyst_L{L}_hx/forward.json",  bwd=f"{ROOT}/hyst_L{L}_hx/backward.json"),
    "hz-sweep  (hx=0)":   dict(field="hz", order="second?",
        fwd=f"{ROOT}/hyst_L{L}_hz/forward.json",  bwd=f"{ROOT}/hyst_L{L}_hz/backward.json"),
}
N_EDGES = lambda L: 3*L**2*(L-1)   # =144 at L=4
LOOP_TOL = 0.05                     # |mag_fwd - mag_bwd| above this counts as 'in the loop'
print("experiments:", list(EXP))

## 2 · Load the two branches

In [ ]:
# ====================== 2 · DATA ======================
def load_branch(path):
    """A check_convergence --dump json -> sorted (h, E, mag). {} if absent."""
    if not os.path.exists(path): return None
    d = json.load(open(path))
    mag = d.get("mx", d.get("mz"))          # whichever conjugate magnetization was dumped
    h = np.array(d["field"], float); o = np.argsort(h)
    return dict(h=h[o], E=np.array(d["E"], float)[o],
                mag=(np.array(mag, float)[o] if mag is not None else None))

DATA = {}
for name, e in EXP.items():
    fwd, bwd = load_branch(e["fwd"]), load_branch(e["bwd"])
    DATA[name] = dict(e, fwd=fwd, bwd=bwd)
    def _n(b): return 'MISSING' if b is None else f"{len(b['h'])} pts [{b['h'].min():.3g},{b['h'].max():.3g}]"
    print(f"[{name}]  forward={_n(fwd)}   backward={_n(bwd)}")

## 3 · Hysteresis plot + loop readout

Top: order parameter (`mag`) — a gap between blue/red is the loop. Bottom: $E/N$ — branches crossing (with the metastable one sitting *above*) is the first-order kink.

In [ ]:
# ====================== 3 · PLOT + METRICS ======================
def aligned(fwd, bwd, key):
    """Values on the fields common to both branches (they share the same grid)."""
    hb = {round(h,4): v for h, v in zip(bwd["h"], bwd[key])}
    h, a, b = [], [], []
    for hi, av in zip(fwd["h"], fwd[key]):
        if round(hi,4) in hb: h.append(hi); a.append(av); b.append(hb[round(hi,4)])
    return np.array(h), np.array(a), np.array(b)

fig, ax = plt.subplots(2, len(DATA), figsize=(7*len(DATA), 9), squeeze=False)
for j, (name, D) in enumerate(DATA.items()):
    fwd, bwd = D["fwd"], D["bwd"]; f = D["field"]
    amag, aE = ax[0, j], ax[1, j]
    if fwd is None or bwd is None:
        for a in (amag, aE): a.text(0.5, 0.5, "branch(es) missing", ha="center", transform=a.transAxes)
        amag.set_title(name); continue
    # --- order parameter (loop) ---
    if fwd["mag"] is not None:
        amag.plot(fwd["h"], fwd["mag"], "o-", color="#1f77b4", ms=5, label="forward (h\u2191)")
        amag.plot(bwd["h"], bwd["mag"], "o-", color="#d62728", ms=5, label="backward (h\u2193)")
        hh, mf, mb = aligned(fwd, bwd, "mag")
        inloop = np.abs(mf - mb) > LOOP_TOL
        loop_w = (hh[inloop].max() - hh[inloop].min()) if inloop.any() else 0.0
        if inloop.any(): amag.axvspan(hh[inloop].min(), hh[inloop].max(), color="gold", alpha=0.15)
        amag.set_title(f"{name}   [{D['order']}]   loop width \u0394{f}={loop_w:.3f}")
    amag.set(xlabel=f"${f}$", ylabel=r"$\langle M\rangle$ (order param)"); amag.legend(fontsize=9)
    # --- energy (crossing) ---
    N = N_EDGES(L)
    aE.plot(fwd["h"], fwd["E"]/N, "o-", color="#1f77b4", ms=5, label="forward")
    aE.plot(bwd["h"], bwd["E"]/N, "o-", color="#d62728", ms=5, label="backward")
    hh, ef, eb = aligned(fwd, bwd, "E")
    gap = np.abs(ef - eb)/N
    aE.set(xlabel=f"${f}$", ylabel="$E/N$",
           title=f"max branch energy gap = {gap.max():.2e} /site" if len(gap) else name)
    aE.legend(fontsize=9)
plt.tight_layout(); plt.show()
print("Loop width > 0 and a nonzero energy gap over a window => first order.")
print("Both ~0 (branches coincide) => second order.")